# A/A Test Results: Deep Analysis

Comprehensive analysis of A/A test results from the shared BQ results table.
Covers:
1. Data quality & coverage
2. Total-level bias assessment
3. Per-segment bias scorecard
4. Uplift distribution analysis
5. Stability across time windows
6. Exact match quality assessment
7. Cross-market comparison (DE vs NL)
8. Statistical tests
9. Recommendations

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

from google.colab import auth, drive
from google.cloud import bigquery

auth.authenticate_user()
drive.mount('/content/drive')
import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)
import aa_config as cfg

client = bigquery.Client(project=cfg.PROJECT_ID)
print(f'Connected to {cfg.PROJECT_ID}')

---
## 2. Load All Results

In [ ]:
df = client.query(f"""
    SELECT *
    FROM `{cfg.AA_RESULTS_TABLE}`
    ORDER BY technique, country, window_start, seed, base_segment
""").to_dataframe()

print(f'Total rows: {len(df)}')
print(f'Techniques: {df["technique"].unique().tolist()}')
print(f'Countries: {df["country"].unique().tolist()}')
print(f'Date range: {df["window_start"].min()} to {df["window_start"].max()}')
print(f'Segments: {sorted(df["base_segment"].unique().tolist())}')
print()

# Summary per technique
for tech in df['technique'].unique():
    sub = df[df['technique']==tech]
    total = sub[sub['base_segment']=='total']
    print(f'{tech}: {len(sub)} rows, {len(total)} total-level runs, '
          f'countries={sub["country"].unique().tolist()}')

---
## 3. Data Quality & Coverage

In [ ]:
# Runs per country x window x technique
pivot_coverage = df[df['base_segment']=='total'].groupby(
    ['technique', 'country', 'window_start']
).size().reset_index(name='n_runs')

print('Runs per technique/country/window:')
print(pivot_coverage.pivot_table(
    index=['country', 'window_start'],
    columns='technique',
    values='n_runs',
    fill_value=0
).to_string())

print(f'\nTotal runs per technique:')
print(pivot_coverage.groupby('technique')['n_runs'].sum().to_string())

In [ ]:
# Segment coverage: how many runs per segment
seg_cov = df.groupby(['technique', 'base_segment']).size().reset_index(name='n_runs')
print('Runs per technique/segment:')
print(seg_cov.pivot_table(
    index='base_segment', columns='technique', values='n_runs', fill_value=0
).to_string())

---
## 4. Total-Level Bias Assessment

In [ ]:
def bias_summary(df_sub, label=''):
    """Compute bias summary statistics for a set of uplift values."""
    vals = df_sub['campaign_period_uplift'].dropna()
    if len(vals) < 2:
        return None
    t_stat, p_val = stats.ttest_1samp(vals, 0)
    rng = np.random.RandomState(42)
    boots = [rng.choice(vals.values, size=len(vals), replace=True).mean()
             for _ in range(10_000)]
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])
    ci_zero = bool(ci_lo <= 0 <= ci_hi)
    abs_mean = abs(vals.mean())
    if abs_mean > cfg.BIAS_THRESHOLD_HARD:
        verdict = 'HARD_FAIL'
    elif abs_mean > cfg.BIAS_THRESHOLD_WARN:
        verdict = 'WARNING'
    elif not ci_zero:
        verdict = 'WARNING'
    else:
        verdict = 'PASS'
    return {
        'label': label, 'n': len(vals),
        'mean': vals.mean(), 'std': vals.std(),
        'median': vals.median(),
        'min': vals.min(), 'max': vals.max(),
        'p_value': p_val,
        'ci_lower': ci_lo, 'ci_upper': ci_hi,
        'ci_contains_zero': ci_zero,
        'verdict': verdict,
    }

# Total-level by technique and country
total = df[df['base_segment']=='total']
rows = []
for tech in sorted(total['technique'].unique()):
    for country in sorted(total[total['technique']==tech]['country'].unique()):
        sub = total[(total['technique']==tech) & (total['country']==country)]
        r = bias_summary(sub, f'{tech} | {country}')
        if r:
            rows.append(r)
    # Also overall for the technique
    r = bias_summary(total[total['technique']==tech], f'{tech} | ALL')
    if r:
        rows.append(r)

df_total_bias = pd.DataFrame(rows)
print('TOTAL-LEVEL BIAS ASSESSMENT')
print('=' * 80)
print(df_total_bias[['label','n','mean','std','median','p_value',
                      'ci_lower','ci_upper','ci_contains_zero','verdict']
    ].to_string(index=False, float_format='%.4f'))

---
## 5. Per-Segment Scorecard

In [ ]:
# Per-segment bias for each technique
seg_rows = []
for tech in sorted(df['technique'].unique()):
    for seg in sorted(df[df['technique']==tech]['base_segment'].unique()):
        sub = df[(df['technique']==tech) & (df['base_segment']==seg)]
        r = bias_summary(sub, f'{seg}')
        if r:
            r['technique'] = tech
            seg_rows.append(r)

df_seg = pd.DataFrame(seg_rows)

for tech in sorted(df_seg['technique'].unique()):
    print(f'\n{"=" * 60}')
    print(f'  {tech} — PER-SEGMENT SCORECARD')
    print(f'{"=" * 60}')
    sub = df_seg[df_seg['technique']==tech].sort_values('mean', key=abs, ascending=False)
    print(sub[['label','n','mean','std','p_value','ci_lower','ci_upper','verdict']
        ].to_string(index=False, float_format='%.4f'))

In [ ]:
# Verdict heatmap
for tech in sorted(df_seg['technique'].unique()):
    sub = df_seg[df_seg['technique']==tech]
    if len(sub) == 0:
        continue

    # Pivot for heatmap
    heat_data = sub.set_index('label')[['mean']].copy()
    heat_data = heat_data.sort_values('mean')

    fig, ax = plt.subplots(figsize=(6, max(len(heat_data)*0.5, 3)))

    # Color by verdict
    colors = []
    for _, row in sub.iterrows():
        if row['verdict'] == 'PASS':
            colors.append('#2ecc71')
        elif row['verdict'] == 'WARNING':
            colors.append('#f39c12')
        else:
            colors.append('#e74c3c')

    sub_sorted = sub.sort_values('mean')
    ax.barh(range(len(sub_sorted)), sub_sorted['mean'].values,
            color=[colors[sub.index.get_loc(i)] for i in sub_sorted.index])
    ax.set_yticks(range(len(sub_sorted)))
    ax.set_yticklabels(sub_sorted['label'].values)
    ax.axvline(x=0, color='black', lw=0.8)
    ax.axvline(x=cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.6)
    ax.axvline(x=-cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.6)
    ax.axvline(x=cfg.BIAS_THRESHOLD_HARD, color='red', ls=':', alpha=0.6)
    ax.axvline(x=-cfg.BIAS_THRESHOLD_HARD, color='red', ls=':', alpha=0.6)
    ax.set_xlabel('Mean Campaign-Period Uplift')
    ax.set_title(f'{tech}: Per-Segment Bias (0 = unbiased)')

    legend_patches = [
        mpatches.Patch(color='#2ecc71', label='PASS (|uplift| ≤ 2%)'),
        mpatches.Patch(color='#f39c12', label='WARNING (2-5%)'),
        mpatches.Patch(color='#e74c3c', label='HARD FAIL (>5%)'),
    ]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 6. Uplift Distribution Analysis

In [ ]:
# Histogram per technique — total level
total = df[df['base_segment']=='total']
techniques = sorted(total['technique'].unique())

fig, axes = plt.subplots(1, max(len(techniques), 1), figsize=(7*len(techniques), 5), squeeze=False)
for i, tech in enumerate(techniques):
    ax = axes[0][i]
    vals = total[total['technique']==tech]['campaign_period_uplift'].dropna()
    if len(vals) == 0:
        ax.set_title(f'{tech} (no data)')
        continue
    sns.histplot(vals, kde=True, ax=ax, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(x=0, color='red', ls='--', lw=2, label='Expected (0)')
    ax.axvline(x=vals.mean(), color='orange', ls='-', lw=2,
               label=f'Mean: {vals.mean():.4f}')
    ax.axvspan(-cfg.BIAS_THRESHOLD_WARN, cfg.BIAS_THRESHOLD_WARN,
               alpha=0.1, color='green', label='PASS zone')
    ax.set_title(f'{tech} — Total Level')
    ax.set_xlabel('Campaign Period Uplift')
    ax.legend(fontsize=9)
plt.suptitle('A/A Test: Uplift Distribution (Total Level)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram per segment for V3 (the main technique)
v3 = df[df['technique']=='V3_CUSTOMER']
segments = sorted(v3['base_segment'].unique())
segments = [s for s in segments if s != 'total']

n_cols = 3
n_rows = (len(segments) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows), squeeze=False)

for i, seg in enumerate(segments):
    ax = axes[i // n_cols][i % n_cols]
    vals = v3[v3['base_segment']==seg]['campaign_period_uplift'].dropna()
    if len(vals) == 0:
        ax.set_title(f'{seg} (no data)')
        continue
    color = '#2ecc71' if abs(vals.mean()) <= 0.02 else '#f39c12' if abs(vals.mean()) <= 0.05 else '#e74c3c'
    sns.histplot(vals, kde=True, ax=ax, bins=15, color=color, edgecolor='white', alpha=0.7)
    ax.axvline(x=0, color='red', ls='--', lw=1.5)
    ax.axvline(x=vals.mean(), color='black', ls='-', lw=1.5)
    ax.set_title(f'{seg} (mean: {vals.mean():.3f})', fontsize=11)
    ax.set_xlabel('Uplift')

# Remove empty subplots
for i in range(len(segments), n_rows * n_cols):
    axes[i // n_cols][i % n_cols].set_visible(False)

plt.suptitle('V3 Customer Lookalike: Uplift Distribution per Segment', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Stability Across Time Windows

In [ ]:
# Box plot: uplift by window — total level per technique
for tech in techniques:
    sub = total[total['technique']==tech]
    if len(sub) < 4:
        continue
    fig, ax = plt.subplots(figsize=(12, 5))
    windows = sorted(sub['window_start'].unique())
    data_by_window = [sub[sub['window_start']==w]['campaign_period_uplift'].dropna().values
                      for w in windows]
    bp = ax.boxplot(data_by_window, labels=windows, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)
    ax.axhline(y=0, color='red', ls='--', lw=1.5)
    ax.axhline(y=cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.axhline(y=-cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.set_title(f'{tech}: Campaign Uplift by Time Window (Total Level)')
    ax.set_xlabel('Window Start')
    ax.set_ylabel('Uplift')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Kruskal-Wallis + Levene
    groups = [g.values for g in [sub[sub['window_start']==w]['campaign_period_uplift'].dropna()
              for w in windows] if len(g) >= 2]
    if len(groups) >= 2:
        lev_stat, lev_p = stats.levene(*groups)
        kw_stat, kw_p = stats.kruskal(*groups)
        print(f'{tech} stability tests:')
        print(f'  Levene (equal variance): stat={lev_stat:.4f}, p={lev_p:.4f} '
              f'— {"WARNING: variance differs" if lev_p < 0.05 else "PASS"}')
        print(f'  Kruskal-Wallis (equal medians): stat={kw_stat:.4f}, p={kw_p:.4f} '
              f'— {"WARNING: medians differ" if kw_p < 0.05 else "PASS"}')

---
## 8. Exact Match Quality

In [ ]:
# Only relevant for V3_CUSTOMER
v3_total = df[(df['technique']=='V3_CUSTOMER') & (df['base_segment']=='total')]

if len(v3_total) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Exact match coverage distribution
    ax = axes[0]
    sns.histplot(v3_total['exact_match_pct'].dropna(), kde=True, ax=ax,
                 bins=20, color='seagreen', edgecolor='white')
    ax.axvline(x=v3_total['exact_match_pct'].mean(), color='orange', ls='-', lw=2,
               label=f'Mean: {v3_total["exact_match_pct"].mean():.1%}')
    ax.set_title('Exact Match Coverage (Total Level)')
    ax.set_xlabel('Exact Match %')
    ax.legend()

    # Exact match % vs uplift — is higher coverage = less bias?
    ax = axes[1]
    ax.scatter(v3_total['exact_match_pct'], v3_total['campaign_period_uplift'],
               alpha=0.5, color='steelblue', edgecolor='white')
    ax.axhline(y=0, color='red', ls='--', lw=1)
    ax.set_xlabel('Exact Match %')
    ax.set_ylabel('Campaign Period Uplift')
    ax.set_title('Does Higher Exact Match Coverage Reduce Bias?')

    # Correlation
    corr_vals = v3_total[['exact_match_pct','campaign_period_uplift']].dropna()
    if len(corr_vals) > 5:
        r, p = stats.pearsonr(corr_vals['exact_match_pct'], corr_vals['campaign_period_uplift'])
        ax.annotate(f'r={r:.3f}, p={p:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                    fontsize=11, ha='left', va='top')

    plt.tight_layout()
    plt.show()

# Per-segment exact match coverage
v3_seg = df[(df['technique']=='V3_CUSTOMER') & (df['base_segment']!='total')]
if len(v3_seg) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    order = sorted(v3_seg['base_segment'].unique())
    sns.boxplot(data=v3_seg, x='base_segment', y='exact_match_pct',
                order=order, ax=ax, palette='Set2')
    ax.set_title('Exact Match Coverage by Segment')
    ax.set_xlabel('Segment')
    ax.set_ylabel('Exact Match %')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

---
## 9. Cross-Market Comparison (DE vs NL)

In [ ]:
# Compare bias between DE and NL for each technique
for tech in techniques:
    sub = total[total['technique']==tech]
    countries = sorted(sub['country'].unique())
    if len(countries) < 2:
        print(f'{tech}: only one country ({countries}), skipping comparison')
        continue

    fig, ax = plt.subplots(figsize=(8, 5))
    data = [sub[sub['country']==c]['campaign_period_uplift'].dropna().values for c in countries]
    bp = ax.boxplot(data, labels=countries, patch_artist=True)
    colors_bp = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
    for j, patch in enumerate(bp['boxes']):
        patch.set_facecolor(colors_bp[j % len(colors_bp)])
        patch.set_alpha(0.6)
    ax.axhline(y=0, color='red', ls='--', lw=1.5)
    ax.set_title(f'{tech}: Campaign Uplift by Country (Total Level)')
    ax.set_ylabel('Uplift')
    plt.tight_layout()
    plt.show()

    # Mann-Whitney U test: is bias different between countries?
    if len(data[0]) >= 3 and len(data[1]) >= 3:
        u_stat, u_p = stats.mannwhitneyu(data[0], data[1], alternative='two-sided')
        print(f'  Mann-Whitney U: stat={u_stat:.1f}, p={u_p:.4f} '
              f'— {"Countries differ significantly" if u_p < 0.05 else "No significant difference"}')
        for c, d in zip(countries, data):
            print(f'    {c}: mean={np.mean(d):.4f}, median={np.median(d):.4f}, n={len(d)}')

---
## 10. Segment-Level Deep Dive: Where Does V3 Break?

In [ ]:
# Scatter: mean uplift vs exact match coverage per segment
v3_seg_summary = df_seg[df_seg['technique']=='V3_CUSTOMER'].copy()

if len(v3_seg_summary) > 0:
    # Get mean exact match pct per segment
    seg_exact = df[df['technique']=='V3_CUSTOMER'].groupby('base_segment')['exact_match_pct'].mean()
    v3_seg_summary = v3_seg_summary.merge(
        seg_exact.reset_index().rename(columns={'exact_match_pct': 'mean_exact_pct'}),
        left_on='label', right_on='base_segment', how='left'
    )

    fig, ax = plt.subplots(figsize=(10, 7))
    for _, row in v3_seg_summary.iterrows():
        color = '#2ecc71' if row['verdict']=='PASS' else '#f39c12' if row['verdict']=='WARNING' else '#e74c3c'
        ax.scatter(row.get('mean_exact_pct', 0), row['mean'], color=color, s=100, zorder=5,
                   edgecolor='white', linewidth=1.5)
        ax.annotate(row['label'], (row.get('mean_exact_pct', 0), row['mean']),
                    fontsize=9, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')

    ax.axhline(y=0, color='black', ls='-', lw=0.5)
    ax.axhline(y=cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.axhline(y=-cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.axhline(y=cfg.BIAS_THRESHOLD_HARD, color='red', ls=':', alpha=0.7)
    ax.axhline(y=-cfg.BIAS_THRESHOLD_HARD, color='red', ls=':', alpha=0.7)
    ax.set_xlabel('Mean Exact Match %')
    ax.set_ylabel('Mean Campaign-Period Uplift')
    ax.set_title('V3: Segment Bias vs Match Quality')

    legend_patches = [
        mpatches.Patch(color='#2ecc71', label='PASS'),
        mpatches.Patch(color='#f39c12', label='WARNING'),
        mpatches.Patch(color='#e74c3c', label='HARD FAIL'),
    ]
    ax.legend(handles=legend_patches, loc='upper left')
    plt.tight_layout()
    plt.show()

    print('\nInterpretation:')
    print('Segments in the top-right (high match %, high bias) have trivial matches')
    print('— everyone matches because features are all near-zero.')
    print('Segments near y=0 regardless of match % are genuinely well-matched.')

---
## 11. Pre-Period vs Campaign-Period Bias Correlation

In [ ]:
# Does pre-period bias predict campaign-period bias?
# If yes, pre-period checks can serve as an early warning.
v3_all = df[df['technique']=='V3_CUSTOMER'].copy()
pre_vs_camp = v3_all[['base_segment','pre_period_uplift','campaign_period_uplift']].dropna()

if len(pre_vs_camp) > 10:
    fig, ax = plt.subplots(figsize=(8, 6))

    for seg in sorted(pre_vs_camp['base_segment'].unique()):
        sub = pre_vs_camp[pre_vs_camp['base_segment']==seg]
        ax.scatter(sub['pre_period_uplift'], sub['campaign_period_uplift'],
                   alpha=0.3, s=20, label=seg if seg in ['total','engaged','churned','lapsing'] else None)

    ax.axhline(y=0, color='red', ls='--', lw=0.8, alpha=0.5)
    ax.axvline(x=0, color='red', ls='--', lw=0.8, alpha=0.5)
    ax.set_xlabel('Pre-Period Uplift')
    ax.set_ylabel('Campaign-Period Uplift')
    ax.set_title('Pre-Period Bias vs Campaign-Period Bias')
    ax.legend(fontsize=8, loc='upper left')

    r, p = stats.pearsonr(pre_vs_camp['pre_period_uplift'], pre_vs_camp['campaign_period_uplift'])
    ax.annotate(f'Overall r={r:.3f}, p={p:.1e}', xy=(0.95, 0.05), xycoords='axes fraction',
                ha='right', fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f'Correlation between pre-period and campaign-period bias: r={r:.3f}')
    if r > 0.5 and p < 0.01:
        print('STRONG correlation — pre-period bias check is a reliable predictor.')
        print('Reject any production run where |pre-period uplift| > threshold.')
    elif r > 0.3:
        print('MODERATE correlation — pre-period check is useful but not definitive.')
    else:
        print('WEAK correlation — pre-period and campaign-period bias are independent.')

---
## 12. Cross-Technique Comparison (if City Lookalike data available)

In [ ]:
city = df[df['technique']=='CITY_BSTS']
v3_t = df[(df['technique']=='V3_CUSTOMER') & (df['base_segment']=='total')]

if len(city) > 0 and len(v3_t) > 0:
    print(f'City Lookalike: {len(city[city["base_segment"]=="total"])} total-level runs')
    print(f'V3 Customer Lookalike: {len(v3_t)} total-level runs')
    print()

    # Side-by-side comparison
    city_total = city[city['base_segment']=='total']

    fig, ax = plt.subplots(figsize=(10, 5))
    data = [v3_t['campaign_period_uplift'].dropna().values,
            city_total['campaign_period_uplift'].dropna().values]
    labels = [f'V3 Customer\n(n={len(data[0])})', f'City BSTS\n(n={len(data[1])})']
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('#3498db')
    bp['boxes'][1].set_facecolor('#e67e22')
    for b in bp['boxes']:
        b.set_alpha(0.6)
    ax.axhline(y=0, color='red', ls='--', lw=1.5)
    ax.axhline(y=cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.5)
    ax.axhline(y=-cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.5)
    ax.set_ylabel('Campaign Period Uplift')
    ax.set_title('A/A Test: V3 Customer vs City Lookalike (Total Level)')
    plt.tight_layout()
    plt.show()

    # Stats
    for label, vals in zip(['V3 Customer', 'City BSTS'], data):
        r = bias_summary(pd.DataFrame({'campaign_period_uplift': vals}), label)
        if r:
            print(f'{label}: mean={r["mean"]:.4f}, std={r["std"]:.4f}, '
                  f'CI=[{r["ci_lower"]:.4f}, {r["ci_upper"]:.4f}], {r["verdict"]}')

    # Mann-Whitney
    if len(data[0]) >= 3 and len(data[1]) >= 3:
        u_stat, u_p = stats.mannwhitneyu(data[0], data[1], alternative='two-sided')
        print(f'\nMann-Whitney U: stat={u_stat:.1f}, p={u_p:.4f}')
        if u_p < 0.05:
            print('Techniques have SIGNIFICANTLY different bias distributions.')
        else:
            print('No significant difference in bias between techniques.')
else:
    print('City Lookalike results not yet available.')
    print('Run aa_city_lookalike.ipynb first, then re-run this analysis.')

---
## 13. Summary & Recommendations

In [ ]:
print('=' * 70)
print('  A/A TEST RESULTS — EXECUTIVE SUMMARY')
print('=' * 70)

for tech in sorted(df['technique'].unique()):
    sub_total = df[(df['technique']==tech) & (df['base_segment']=='total')]
    tech_segs = df_seg[df_seg['technique']==tech]
    n_pass = len(tech_segs[tech_segs['verdict']=='PASS'])
    n_warn = len(tech_segs[tech_segs['verdict']=='WARNING'])
    n_fail = len(tech_segs[tech_segs['verdict']=='HARD_FAIL'])

    vals = sub_total['campaign_period_uplift'].dropna()
    print(f'\n--- {tech} ---')
    print(f'  Runs: {len(vals)}')
    if len(vals) > 0:
        print(f'  Total-level mean uplift: {vals.mean():+.4f} ({vals.mean():+.1%})')
        print(f'  Total-level std: {vals.std():.4f}')
    print(f'  Segments: {n_pass} PASS, {n_warn} WARNING, {n_fail} HARD FAIL')

    # List failing segments
    failing = tech_segs[tech_segs['verdict']=='HARD_FAIL']['label'].tolist()
    if failing:
        print(f'  Failing segments: {", ".join(failing)}')

    passing = tech_segs[tech_segs['verdict']=='PASS']['label'].tolist()
    if passing:
        print(f'  Passing segments: {", ".join(passing)}')

print(f'\n{"=" * 70}')
print('  RECOMMENDATIONS')
print(f'{"=" * 70}')

v3_segs = df_seg[df_seg['technique']=='V3_CUSTOMER']
if len(v3_segs) > 0:
    passing = v3_segs[v3_segs['verdict']=='PASS']['label'].tolist()
    failing = v3_segs[v3_segs['verdict']=='HARD_FAIL']['label'].tolist()
    warning = v3_segs[v3_segs['verdict']=='WARNING']['label'].tolist()

    print(f'\nV3 Customer Lookalike:')
    if passing:
        print(f'  USE with confidence for: {", ".join(passing)}')
    if warning:
        print(f'  USE with caution for: {", ".join(warning)} (apply bias correction)')
    if failing:
        print(f'  DO NOT USE for: {", ".join(failing)} (bias > 5%)')
        print(f'  → Use City Lookalike or Clustered Geo for these segments instead')

if len(city) > 0:
    city_total_vals = city[city['base_segment']=='total']['campaign_period_uplift'].dropna()
    if len(city_total_vals) > 0:
        city_verdict = 'PASS' if abs(city_total_vals.mean()) <= 0.02 else 'WARNING' if abs(city_total_vals.mean()) <= 0.05 else 'HARD_FAIL'
        print(f'\nCity Lookalike (BSTS):')
        print(f'  Total-level: {city_total_vals.mean():+.4f} — {city_verdict}')